# Quick linearized focus model demo

This is intentionally small and fast. It trains a simple classifier on a balanced sample of the final linearized dataset, checks validation performance, saves a demo prediction pipeline, and tries a few hand-made room-condition examples. This is not the final evaluation setup; real-world testing can use a stricter split later.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

MAL_DIR = Path.cwd()
if MAL_DIR.name != "MAL":
    MAL_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if path.name == "MAL")

DATASET_PATH = MAL_DIR / "data" / "processed" / "linearized_session_windows_30min.csv"
MODEL_PATH = MAL_DIR / "models" / "quick_linearized_focus_model.joblib"
TARGET_COLUMN = "focus_score"
RANDOM_STATE = 42
MAX_ROWS_PER_CLASS = 8_000

DATASET_PATH, MODEL_PATH

(PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/data/processed/linearized_session_windows_30min.csv'), PosixPath('/home/edf/Documents/GitSynced/SEP4_/MAL/models/quick_linearized_focus_model.joblib'))

## Load a balanced training sample

In [2]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

balanced_parts = []
for focus_score, group in df.groupby(TARGET_COLUMN):
    balanced_parts.append(
        group.sample(
            n=min(len(group), MAX_ROWS_PER_CLASS),
            random_state=RANDOM_STATE,
        )
    )

demo_df = pd.concat(balanced_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
feature_columns = [column for column in demo_df.columns if column != TARGET_COLUMN]

sample_summary = pd.DataFrame([
    {
        "full_rows": len(df),
        "demo_rows": len(demo_df),
        "features": len(feature_columns),
        "target_min": int(demo_df[TARGET_COLUMN].min()),
        "target_max": int(demo_df[TARGET_COLUMN].max()),
    }
])
sample_summary

full_rows,demo_rows,features,target_min,target_max
866348,40000,35,1,5


In [3]:
demo_df[TARGET_COLUMN].value_counts().sort_index().rename("rows_per_score")

focus_score
1    8000
2    8000
3    8000
4    8000
5    8000

## Train a quick prediction pipeline

In [4]:
X = demo_df[feature_columns]
y = demo_df[TARGET_COLUMN].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    ExtraTreesClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
)

model.fit(X_train, y_train)
validation_predictions = model.predict(X_val)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": model, "feature_columns": feature_columns}, MODEL_PATH)

metrics = pd.DataFrame([
    {
        "accuracy": accuracy_score(y_val, validation_predictions),
        "macro_f1": f1_score(y_val, validation_predictions, average="macro"),
        "mean_absolute_error": mean_absolute_error(y_val, validation_predictions),
        "within_1_score": (abs(validation_predictions - y_val) <= 1).mean(),
        "balanced_random_accuracy_baseline": 1 / y.nunique(),
        "saved_model": str(MODEL_PATH),
    }
])
metrics

accuracy,macro_f1,mean_absolute_error,within_1_score,balanced_random_accuracy_baseline,saved_model
0.24675,0.241228,1.512875,0.546875,0.2,/home/edf/Documents/GitSynced/SEP4_/MAL/models/quick_linearized_focus_model.joblib


In [5]:
print(classification_report(y_val, validation_predictions, zero_division=0))

              precision    recall  f1-score   support

           1       0.31      0.41      0.36      1600
           2       0.22      0.22      0.22      1600
           3       0.22      0.20      0.21      1600
           4       0.23      0.16      0.19      1600
           5       0.22      0.24      0.23      1600

    accuracy                           0.25      8000
   macro avg       0.24      0.25      0.24      8000
weighted avg       0.24      0.25      0.24      8000


## Try a few obvious room conditions

These examples do not have ground-truth labels. They are a smoke test: comfortable conditions should not look terrible, and clearly bad hot/stuffy/noisy conditions should land lower.

In [6]:
def make_window_row(temperature, humidity, light, noise, co2, range_factor=0.05, count=7):
    values = {}
    for feature, latest in {
        "temperature": temperature,
        "humidity": humidity,
        "light": light,
        "noise": noise,
        "co2": co2,
    }.items():
        spread = abs(latest) * range_factor if pd.notna(latest) else 0
        values[f"{feature}_latest"] = latest
        values[f"{feature}_mean"] = latest
        values[f"{feature}_min"] = latest - spread if pd.notna(latest) else pd.NA
        values[f"{feature}_max"] = latest + spread if pd.notna(latest) else pd.NA
        values[f"{feature}_std"] = spread / 2
        values[f"{feature}_count"] = count if pd.notna(latest) else 0
        values[f"{feature}_range"] = spread * 2 if pd.notna(latest) else pd.NA
    return values

obvious_examples = pd.DataFrame(
    [
        make_window_row(22, 45, 500, 42, 650),
        make_window_row(29, 70, 80, 75, 1800),
        make_window_row(18, 35, 100, 60, 1200),
        make_window_row(24, 50, 400, 50, 900),
    ],
    index=["comfortable", "bad_hot_stuffy_noisy", "cold_dim_middling", "okay"],
).reindex(columns=feature_columns)

example_predictions = pd.DataFrame(
    {
        "example": obvious_examples.index,
        "predicted_focus_score": model.predict(obvious_examples),
    }
)
example_predictions

example,predicted_focus_score
comfortable,4
bad_hot_stuffy_noisy,1
cold_dim_middling,3
okay,4


In [7]:
comfortable_score = int(example_predictions.loc[example_predictions["example"].eq("comfortable"), "predicted_focus_score"].iloc[0])
bad_score = int(example_predictions.loc[example_predictions["example"].eq("bad_hot_stuffy_noisy"), "predicted_focus_score"].iloc[0])
sanity_check = pd.DataFrame([
    {
        "comfortable_prediction": comfortable_score,
        "bad_hot_stuffy_noisy_prediction": bad_score,
        "comfortable_is_at_least_as_good_as_bad": comfortable_score >= bad_score,
    }
])
sanity_check

comfortable_prediction,bad_hot_stuffy_noisy_prediction,comfortable_is_at_least_as_good_as_bad
4,1,True
